<a href="https://colab.research.google.com/github/marcplanas11-alt/insurance-claims-triage-ai/blob/main/claims_agentic_workflow_v2_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Decision agent

In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 18.0 MB/s eta 0:00:00


In [2]:
import anthropic
import json

In [7]:
from getpass import getpass

ANTHROPIC_API_KEY = getpass("Enter your Anthropic API key: ")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

Enter your Anthropic API key: ··········


In [8]:
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Reply with only this JSON: {\"status\": \"ok\"}"
        }
    ]
)

print(response.content[0].text)

```json
{"status": "ok"}
```


reconstruir el state mínimo


In [9]:
claim_text = """
The insured reports water damage to the kitchen caused by a burst pipe.
Estimated repair cost is £8,500.
The incident occurred 18 days ago.
The insured has provided photos but no plumber invoice yet.
"""

policy_text = """
The policy covers sudden and accidental escape of water from fixed domestic installations.
Claims must be notified as soon as reasonably practicable.
Gradual damage, wear and tear, and pre-existing damage are excluded.
The insurer may request invoices, photos, and repair reports.
"""

state = {
    "claim_text": claim_text,
    "policy_text": policy_text,
    "intake": {
        "claim_type": "water_damage",
        "cause": "burst_pipe",
        "estimated_loss": 8500,
        "risk_flags": ["late_notification", "missing_documentation"]
    },
    "coverage_analysis": {
        "coverage_position": "potentially_covered",
        "covered_peril": "escape_of_water",
        "exclusions": ["wear_and_tear", "gradual_damage"],
        "conditions": ["prompt_notification_required"],
        "missing_documents": ["plumber_invoice"],
        "coverage_confidence": 0.75
    },
    "decision": None,
    "confidence": None,
    "risk_flags": ["late_notification", "missing_documentation"]
}

state

{'claim_text': '\nThe insured reports water damage to the kitchen caused by a burst pipe.\nEstimated repair cost is £8,500.\nThe incident occurred 18 days ago.\nThe insured has provided photos but no plumber invoice yet.\n',
 'policy_text': '\nThe policy covers sudden and accidental escape of water from fixed domestic installations.\nClaims must be notified as soon as reasonably practicable.\nGradual damage, wear and tear, and pre-existing damage are excluded.\nThe insurer may request invoices, photos, and repair reports.\n',
 'intake': {'claim_type': 'water_damage',
  'cause': 'burst_pipe',
  'estimated_loss': 8500,
  'risk_flags': ['late_notification', 'missing_documentation']},
 'coverage_analysis': {'coverage_position': 'potentially_covered',
  'covered_peril': 'escape_of_water',
  'exclusions': ['wear_and_tear', 'gradual_damage'],
  'conditions': ['prompt_notification_required'],
  'missing_documents': ['plumber_invoice'],
  'coverage_confidence': 0.75},
 'decision': None,
 'confi

decision_agent



In [10]:
def decision_agent_claude(client, state):
    prompt = f"""
You are an insurance claims decision agent.

You must decide one of:
- APPROVE
- ESCALATE
- REJECT

Return ONLY valid JSON with this exact schema:

{{
  "decision": "APPROVE | ESCALATE | REJECT",
  "confidence": 0.0,
  "risk_flags": [],
  "human_review_required": true,
  "reason": ""
}}

CLAIM:
{state["claim_text"]}

POLICY:
{state["policy_text"]}

INTAKE:
{json.dumps(state["intake"], indent=2)}

COVERAGE_ANALYSIS:
{json.dumps(state["coverage_analysis"], indent=2)}

CURRENT_RISK_FLAGS:
{json.dumps(state["risk_flags"], indent=2)}
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_output = response.content[0].text
    decision = json.loads(raw_output)

    state["decision"] = decision
    state["confidence"] = decision["confidence"]
    state["risk_flags"] = sorted(list(set(state["risk_flags"] + decision["risk_flags"])))

    return state

In [12]:
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": "Return ONLY valid JSON: {\"status\": \"ok\"}"
        }
    ]
)

raw_output = response.content[0].text
print(repr(raw_output))

'```json\n{"status": "ok"}\n```'


In [14]:
import json

def decision_agent_claude(client, state):
    prompt = f"""
You are an insurance claims decision agent.

You must decide one of:
- APPROVE
- ESCALATE
- REJECT

Return ONLY valid JSON with this exact schema:

{{
  "decision": "APPROVE | ESCALATE | REJECT",
  "confidence": 0.0,
  "risk_flags": [],
  "human_review_required": true,
  "reason": ""
}}

CLAIM:
{state["claim_text"]}

POLICY:
{state["policy_text"]}

INTAKE:
{json.dumps(state["intake"], indent=2)}

COVERAGE_ANALYSIS:
{json.dumps(state["coverage_analysis"], indent=2)}

CURRENT_RISK_FLAGS:
{json.dumps(state["risk_flags"], indent=2)}
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_output = response.content[0].text
    # Extract JSON string by finding the first '{' and last '}'
    start_index = raw_output.find('{')
    end_index = raw_output.rfind('}')

    if start_index != -1 and end_index != -1 and start_index < end_index:
        json_string = raw_output[start_index : end_index + 1]
    else:
        # Fallback if the curly braces are not found as expected, try stripping common markdown fences.
        json_string = raw_output.strip()
        if json_string.startswith("```json"):
            json_string = json_string[len("```json"):]
        elif json_string.startswith("```"):
            json_string = json_string[len("```"):]
        if json_string.endswith("```"):
            json_string = json_string[:-len("```")]
        json_string = json_string.strip() # Remove any remaining whitespace/newlines

    decision = json.loads(json_string)

    state["decision"] = decision
    state["confidence"] = decision["confidence"]
    state["risk_flags"] = sorted(list(set(state["risk_flags"] + decision["risk_flags"])))

    return state

state = decision_agent_claude(client, state)

state["decision"]

{'decision': 'ESCALATE',
 'confidence': 0.65,
 'risk_flags': ['late_notification',
  'missing_documentation',
  'notification_delay_18_days'],
 'human_review_required': True,
 'reason': "The claim meets the covered peril (burst pipe/escape of water) and falls within policy scope, but presents two significant concerns requiring human review: 1) 18-day notification delay potentially breaches 'as soon as reasonably practicable' requirement, though may have valid explanation; 2) Missing plumber invoice prevents verification that damage was sudden/accidental vs. gradual/wear-and-tear. Photos provided are positive, but professional assessment documentation needed to exclude policy exclusions. Estimated loss of £8,500 is moderate. Recommend escalation to adjuster to evaluate notification delay reasonableness and request plumber report before making coverage determination."}

In [15]:
print(repr(raw_output))

'```json\n{"status": "ok"}\n```'


In [16]:
def safe_json_parse(raw_output):
    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        return {
            "decision": "ESCALATE",
            "confidence": 0.0,
            "risk_flags": ["invalid_json_output"],
            "human_review_required": True,
            "reason": f"Model did not return valid JSON. Raw output: {raw_output[:300]}"
        }

In [17]:
bad_output = "Here is my answer: ESCALATE because documents are missing"

safe_json_parse(bad_output)

{'decision': 'ESCALATE',
 'confidence': 0.0,
 'risk_flags': ['invalid_json_output'],
 'human_review_required': True,
 'reason': 'Model did not return valid JSON. Raw output: Here is my answer: ESCALATE because documents are missing'}

In [19]:
decision = safe_json_parse(raw_output)

In [20]:
state = decision_agent_claude(client, state)
state["decision"]

{'decision': 'ESCALATE',
 'confidence': 0.65,
 'risk_flags': ['late_notification',
  'missing_documentation',
  'notification_delay_18_days',
  'no_professional_assessment'],
 'human_review_required': True,
 'reason': "Claim requires human review due to: (1) 18-day notification delay which may breach 'as soon as reasonably practicable' requirement - justification needed; (2) Missing plumber invoice to verify cause was sudden/accidental vs. gradual deterioration or wear and tear; (3) Without professional assessment, cannot confirm burst pipe cause and rule out excluded perils. Photos provided are positive but insufficient. Peril is potentially covered and amount (£8,500) is moderate. Recommend requesting plumber report and written explanation for notification delay before final decision."}